In [4]:
"""
TRACK 2 — MARBERT Fine-tuning for Arabic Aspect-Based Sentiment
Dataset: DeepX Arabic reviews (train_flat_balanced.csv / val_flat.csv)
Model:   UBC-NLP/MARBERT  (pretrained on 1B Arabic tweets — MSA + dialectal)

Install:
    pip install transformers datasets accelerate torch scipy
"""

import json
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import re

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
MODEL_NAME = "UBC-NLP/MARBERT"
MAX_LEN    = 128
BATCH_SIZE = 16
EPOCHS     = 5
LR         = 2e-5

# Paths — update if your files live elsewhere
TRAIN_PATH   = "/kaggle/input/datasets/imaisalah/processed-data/train_flat_balanced.csv"   # balanced per-aspect rows
VAL_PATH     = "/kaggle/input/datasets/imaisalah/processed-data/val_flat.csv"              # per-aspect validation rows
WEIGHTS_PATH = "/kaggle/input/datasets/imaisalah/processed-data/sentiment_class_weights.json"

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
# Preprocessing already cleaned the text; we apply a light guard
# only for any residual URLs / mentions that slipped through.
def light_clean(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)

# The preprocessed column is already clean; apply light guard just in case
train_df["clean_text"] = train_df["text_clean"].apply(light_clean)
val_df["clean_text"]   = val_df["text_clean"].apply(light_clean)

print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)}")
print("Label distribution (train):\n", train_df["sentiment"].value_counts())

# ─────────────────────────────────────────────
# 2. ENCODE LABELS
#    sentiment column: 'positive' | 'negative' | 'neutral'
# ─────────────────────────────────────────────
# Fix label order so class-weight indices stay consistent
LABEL_ORDER = ["negative", "neutral", "positive"]   # alphabetical = LabelEncoder default

le = LabelEncoder()
le.fit(LABEL_ORDER)                          # enforce consistent order
train_df["label_id"] = le.transform(train_df["sentiment"])
val_df["label_id"]   = le.transform(val_df["sentiment"])

num_labels = len(le.classes_)
print(f"\nLabel mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# ─────────────────────────────────────────────
# 3. CLASS WEIGHTS  (from sentiment_class_weights.json)
# ─────────────────────────────────────────────
with open(WEIGHTS_PATH) as f:
    raw_weights = json.load(f)

# Build weight tensor in the same order as le.classes_
weight_list = [raw_weights[cls] for cls in le.classes_]
class_weights = torch.tensor(weight_list, dtype=torch.float)
print(f"\nClass weights ({list(le.classes_)}): {weight_list}")

# ─────────────────────────────────────────────
# 4. TOKENIZER + DATASET
# ─────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    )

def make_dataset(dataframe: pd.DataFrame) -> Dataset:
    ds = Dataset.from_dict({
        "clean_text": dataframe["clean_text"].tolist(),
        "labels":     dataframe["label_id"].tolist(),
    })
    return ds.map(tokenize, batched=True)

train_ds = make_dataset(train_df)
val_ds   = make_dataset(val_df)

# ─────────────────────────────────────────────
# 5. MODEL  (weighted loss via custom Trainer)
# ─────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="single_label_classification",
)

class WeightedTrainer(Trainer):
    """Trainer that applies per-class weights to cross-entropy loss."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ─────────────────────────────────────────────
# 6. METRICS
# ─────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1    = f1_score(labels, preds, average="macro",    zero_division=0)
    weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)
    return {"macro_f1": macro_f1, "weighted_f1": weighted_f1}

# ─────────────────────────────────────────────
# 7. TRAINING ARGS
# ─────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./marbert_output",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",        # ← was evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# ─────────────────────────────────────────────
# 8. TRAIN + EVALUATE
# ─────────────────────────────────────────────
trainer.train()
results = trainer.evaluate()
print("\nMARBERT eval results:", results)

# Full classification report on val set
raw_preds = trainer.predict(val_ds)
preds = np.argmax(raw_preds.predictions, axis=1)
print("\nClassification report:")
print(classification_report(
    val_df["label_id"].values, preds,
    target_names=le.classes_, zero_division=0
))

# ─────────────────────────────────────────────
# 9. SAVE PROBABILITIES FOR ENSEMBLE
# ─────────────────────────────────────────────
from scipy.special import softmax
marbert_probs = softmax(raw_preds.predictions, axis=1)   # shape: (N, 3)
np.save("marbert_probs.npy", marbert_probs)

# Also save the label encoder order so the ensemble can align columns
with open("label_encoder_classes.json", "w") as f:
    json.dump(list(le.classes_), f)

print("\nSaved: marbert_probs.npy  +  label_encoder_classes.json")
print("Pass both to the ensemble script.")


Train rows: 4043 | Val rows: 840
Label distribution (train):
 sentiment
positive    1816
negative    1774
neutral      453
Name: count, dtype: int64

Label mapping: {np.str_('negative'): np.int64(0), np.str_('neutral'): np.int64(1), np.str_('positive'): np.int64(2)}

Class weights ([np.str_('negative'), np.str_('neutral'), np.str_('positive')]): [0.7597, 2.975, 0.7421]


Map:   0%|          | 0/4043 [00:00<?, ? examples/s]

Map:   0%|          | 0/840 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Macro F1,Weighted F1
0.191579,0.497078,3,0.699953,0.880613



MARBERT eval results: {'eval_loss': 0.4970777928829193, 'eval_macro_f1': 0.699953371801597, 'eval_weighted_f1': 0.8806133620318792}



Classification report:
              precision    recall  f1-score   support

    negative       0.89      0.91      0.90       357
     neutral       0.47      0.20      0.28        40
    positive       0.90      0.94      0.92       443

    accuracy                           0.89       840
   macro avg       0.76      0.68      0.70       840
weighted avg       0.88      0.89      0.88       840


Saved: marbert_probs.npy  +  label_encoder_classes.json
Pass both to the ensemble script.


In [5]:
# ─────────────────────────────────────────────

# 10. INFERENCE ON TEST SET → submission.json
# ─────────────────────────────────────────────
import json
from scipy.special import softmax

TEST_PATH   = "/kaggle/input/datasets/imaisalah/processed-data/unlabeled_clean.csv"   # update path
OUTPUT_PATH = "submission.json"

# ── Aspect keywords (Arabic) ──────────────────
# Extend these lists with any domain terms you see in your data
ASPECT_KEYWORDS = {
    "food":     ["أكل", "طعام", "وجبة", "مذاق", "طبق", "سندوتش", "بيتزا", "برجر", "طازج", "لذيذ", "مأكولات"],
    "service":  ["خدمة", "نادل", "موظف", "استقبال", "تعامل", "اهتمام", "طاقم", "ودود", "بطيء", "سريع"],
    "delivery": ["توصيل", "ديليفري", "تأخير", "وصل", "شحن", "مندوب"],
    "ambiance": ["جو", "ديكور", "مكان", "هدوء", "نظافة", "تصميم", "إضاءة", "راحة", "جلسة", "أجواء"],
    "price":    ["سعر", "أسعار", "غالي", "رخيص", "قيمة", "تكلفة", "فاتورة"],
}

def detect_aspects(text: str) -> list[str]:
    """Return aspects whose keywords appear in the review text."""
    found = []
    for aspect, keywords in ASPECT_KEYWORDS.items():
        if any(kw in text for kw in keywords):
            found.append(aspect)
    # Fallback: if nothing matched, assign 'food' as the default for restaurant reviews
    return found if found else ["food"]

# ── Load & clean test data ────────────────────
test_df = pd.read_csv(TEST_PATH)
test_df["clean_text"] = test_df["text_clean"].apply(light_clean)   # same cleaner as training

# ── Build per-aspect rows for the model ───────
records = []
for _, row in test_df.iterrows():
    aspects = detect_aspects(row["clean_text"])
    for aspect in aspects:
        records.append({
            "review_id":  row["review_id"],
            "aspect":     aspect,
            "clean_text": row["clean_text"],
        })

infer_df = pd.DataFrame(records)
print(f"Test reviews: {len(test_df)} → expanded to {len(infer_df)} aspect rows")

# ── Tokenize ──────────────────────────────────
infer_ds = Dataset.from_dict({"clean_text": infer_df["clean_text"].tolist(),
                               "labels":     [0] * len(infer_df)})   # dummy labels
infer_ds = infer_ds.map(tokenize, batched=True)

# ── Predict ───────────────────────────────────
raw_preds   = trainer.predict(infer_ds)
probs       = softmax(raw_preds.predictions, axis=1)   # (N, 3)
pred_ids    = np.argmax(probs, axis=1)
pred_labels = le.inverse_transform(pred_ids)           # back to 'negative'/'neutral'/'positive'

infer_df["sentiment"] = pred_labels

# ── Aggregate back to review level ────────────
submission = []
for review_id, group in infer_df.groupby("review_id", sort=False):
    aspects = group["aspect"].tolist()
    aspect_sentiments = dict(zip(group["aspect"], group["sentiment"]))
    submission.append({
        "review_id":         int(review_id),
        "aspects":           aspects,
        "aspect_sentiments": aspect_sentiments,
    })

# Sort by review_id to match expected order
submission.sort(key=lambda x: x["review_id"])

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(submission)} reviews → {OUTPUT_PATH}")
print("Sample:", json.dumps(submission[0], ensure_ascii=False, indent=2))

Test reviews: 7047 → expanded to 7352 aspect rows


Map:   0%|          | 0/7352 [00:00<?, ? examples/s]


Saved 7047 reviews → submission.json
Sample: {
  "review_id": 1,
  "aspects": [
    "food"
  ],
  "aspect_sentiments": {
    "food": "negative"
  }
}
